# 02 - Regression: Opportunity Amount (log1p)

Pipeline:
1. Continuous OptBinning transformer
2. LinearRegression

Target is `log1p(Opportunity Amount USD)`; training data from `../../data/intermediate/df_train_stratified.parquet`.

In [1]:
import numpy as np
import pandas as pd

from optbinning import ContinuousOptimalBinning

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option('display.max_columns', 200)

In [2]:
class ContinuousOptimalBinningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variable_names):
        self.variable_names = variable_names

    def fit(self, X, y):
        self.variable_names_ = list(self.variable_names)
        self.models_ = {}
        self.failed_features_ = []

        for col in self.variable_names_:
            s = X[col]
            dtype = 'numerical' if (pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)) else 'categorical'
            try:
                model = ContinuousOptimalBinning(name=col, dtype=dtype)
                model.fit(s, y)
                self.models_[col] = model
            except Exception:
                self.models_[col] = None
                self.failed_features_.append(col)

        return self

    def transform(self, X):
        out = pd.DataFrame(index=X.index)
        for col in self.variable_names_:
            model = self.models_.get(col)
            if model is None:
                out[col] = 0.0
            else:
                vals = model.transform(X[col], metric='mean')
                out[col] = pd.Series(vals, index=X.index).astype(float)

        return out.fillna(0.0).to_numpy()


In [3]:
df = pd.read_parquet('../../data/intermediate/df_train_stratified.parquet')
print('shape:', df.shape)

shape: (54617, 37)


In [4]:
# Editable feature list (all enabled by default)
FEATURES = [
    'Supplies Group',
    'Supplies Subgroup',
    'Region',
    'Route To Market',
    'Elapsed Days In Sales Stage',
    'Opportunity Result',
    'Opportunity Result Bool',
    'Sales Stage Change Count',
    'Total Days Identified Through Closing',
    'Total Days Identified Through Qualified',
    'Client Size By Revenue (USD)',
    'Client Size By Employee Count',
    'Revenue From Client Past Two Years (USD)',
    'Competitor Type',
    'Ratio Days Identified To Total Days',
    'Ratio Days Validated To Total Days',
    'Ratio Days Qualified To Total Days',
    'Deal Size Category (USD)',
    'total_days_zero',
    'opportunity_amount_weirdness',
    'flag_0_days',
    'flag_ratio_problem',
    'flag_zero_opportunity_amount',
    'flag_outlier_opportunity_amount',
    'flag_outlier_total_days',
    'flag_totally_repeated_row',
    'flag_partially_repeated_row',
    'partial_repeat_is_latest_id_appearance',
    'flag_only_identified',
    'flag_weirdness_over_75th_pct',
    'problem_count',
]

TARGET_RAW = 'Opportunity Amount USD'

missing = [c for c in FEATURES + [TARGET_RAW] if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

X = df[FEATURES].copy()
y_raw = df[TARGET_RAW].astype(float).copy()
y = np.log1p(y_raw)

print('n_features:', len(FEATURES))
print('target raw mean:', y_raw.mean())
print('target log mean:', y.mean())

n_features: 31
target raw mean: 91610.5492246004
target log mean: 10.314419755522284


In [5]:
X_train, X_valid, y_train, y_valid, y_raw_train, y_raw_valid = train_test_split(
    X, y, y_raw, test_size=0.2, random_state=42
)

reg_pipe = Pipeline(steps=[
    ('optbinning', ContinuousOptimalBinningTransformer(variable_names=FEATURES)),
    ('model', LinearRegression()),
])

reg_pipe.fit(X_train, y_train)

failed = reg_pipe.named_steps['optbinning'].failed_features_
print('failed binning features:', failed)

failed binning features: []


In [6]:
yhat_log = reg_pipe.predict(X_valid)

r2_log = r2_score(y_valid, yhat_log)
rmse_log = np.sqrt(mean_squared_error(y_valid, yhat_log))
mae_log = mean_absolute_error(y_valid, yhat_log)

yhat_raw = np.expm1(yhat_log)
yhat_raw = np.clip(yhat_raw, a_min=0, a_max=None)
mae_raw = mean_absolute_error(y_raw_valid, yhat_raw)

print('R2 (log scale):', r2_log)
print('RMSE (log scale):', rmse_log)
print('MAE (log scale):', mae_log)
print('MAE (raw amount):', mae_raw)

R2 (log scale): 0.9098601722312769
RMSE (log scale): 0.6876517437092653
MAE (log scale): 0.4145671159857823
MAE (raw amount): 27776.40190029176
